##**Myanmar Part-of-Speech Tagging using CRF**

**Name:** Han Zaw Htet  
**Dataset:** [myPOS corpus-ver-3.0](https://github.com/ye-kyaw-thu/myPOS/tree/master/corpus-ver-3.0/corpus)
**Tagging helper:** [myWord_tagger.py](https://github.com/ye-kyaw-thu/AIE-F-B2/blob/main/codes/class-5/myWord_tagger.py)

## 1. Install and import libraries

In [31]:
%pip install -q sklearn-crfsuite scikit-learn tabulate

Note: you may need to restart the kernel to use updated packages.


In [32]:
from pathlib import Path
from collections import Counter

import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split
from tabulate import tabulate

print("Libraries ready.")

Libraries ready.


## 2. Locate myPOS dataset


In [33]:
candidates = [
    Path("mypos-ver.3.0.shuf.nopipe.txt"),
    Path("mypos-ver.3.0.shuf.txt"),
]

CORPUS_FILE = next((p for p in candidates if p.exists()), None)

if CORPUS_FILE is None:
    print("Dataset not found. Files in current folder:")
    for p in Path.cwd().iterdir():
        print(" -", p.name)
else:
    print(f"Using corpus file: {CORPUS_FILE.name}")

Using corpus file: mypos-ver.3.0.shuf.nopipe.txt


## 3. Preview raw corpus lines

In [34]:
if CORPUS_FILE is None:
    raise FileNotFoundError(
        "Add myPOS file."
    )

with CORPUS_FILE.open(encoding="utf-8") as fh:
    for i, line in enumerate(fh):
        if i >= 5:
            break
        print(f"[{i + 1}] {line.strip()}\n")

[1] ၁၉၆၂/num ခုနှစ်/n ခန့်မှန်း/v သန်းခေါင်စာရင်း/n အရ/ppm လူဦးရေ/n ၁၁၅၉၃၁/num ယောက်/part ရှိ/v သည်/ppm ။/punc

[2] လူ/n တိုင်း/part တွင်/ppm သင့်မြတ်/v လျော်ကန်/v စွာ/part ကန့်သတ်/v ထား/part သည့်/part အလုပ်/n လုပ်/v ချိန်/n အပြင်/conj ၊/punc လစာ/n နှင့်တကွ/conj အခါ/n ကာလ/n အားလျော်စွာ/ppm သတ်မှတ်/v ထား/part သည့်/part အလုပ်/n အားလပ်ရက်/n များ/part ပါဝင်/v သည့်/part အနားယူခွင့်/n နှင့်/conj အားလပ်ခွင့်/n ခံစားပိုင်ခွင့်/n ရှိ/v သည်/ppm ။/punc

[3] ဤ/adj နည်း/n ကို/ppm စစ်ယူ/v သော/part နည်း/n ဟု/part ခေါ်/v သည်/ppm ။/punc

[4] စာပြန်ပွဲ/n ဆို/v တာ/part က/ppm အာဂုံဆောင်/v အလွတ်ကျက်/v ထား/part တဲ့/part ပိဋကတ်သုံးပုံ/n စာပေ/n တွေ/part ကို/ppm စာစစ်/v သံဃာတော်ကြီး/n တွေ/part ရဲ့/ppm ရှေ့/n မှာ/ppm အလွတ်/adv ပြန်/v ပြီး/part ရွတ်ပြ/v ရ/part တာ/part ပေါ့/part ။/punc

[5] ဒီ/pron မှာ/ppm ကျွန်တော့်/pron သက်သေခံကတ်/n ပါ/part ။/punc



## 4. Parse POS-tagged sentences

Each token is `word/POS`. Pipe `|` in compound forms is treated as a space.

In [35]:
def read_mypos(path):
    sentences = []
    skipped = 0

    with open(path, encoding="utf-8") as fh:
        for raw in fh:
            text = raw.strip().replace("|", " ")
            if not text:
                continue

            sent = []
            for tok in text.split():
                if "/" not in tok:
                    skipped += 1
                    continue
                w, t = tok.rsplit("/", 1)
                w, t = w.strip(), t.strip()
                if w and t:
                    sent.append((w, t))
                else:
                    skipped += 1

            if sent:
                sentences.append(sent)

    return sentences, skipped


corpus, n_skipped = read_mypos(CORPUS_FILE)

print(f"Sentences loaded : {len(corpus):,}")
print(f"Tokens skipped   : {n_skipped:,}")
print("\nSample sentence:")
print(corpus[0] if corpus else "(empty)")

Sentences loaded : 43,196
Tokens skipped   : 0

Sample sentence:
[('၁၉၆၂', 'num'), ('ခုနှစ်', 'n'), ('ခန့်မှန်း', 'v'), ('သန်းခေါင်စာရင်း', 'n'), ('အရ', 'ppm'), ('လူဦးရေ', 'n'), ('၁၁၅၉၃၁', 'num'), ('ယောက်', 'part'), ('ရှိ', 'v'), ('သည်', 'ppm'), ('။', 'punc')]


## 5. Corpus statistics

In [36]:
if not corpus:
    raise ValueError("No sentences were parsed from the corpus.")

tag_list = [tag for sent in corpus for _, tag in sent]
tag_freq = Counter(tag_list)

print(f"Sentences     : {len(corpus):,}")
print(f"Tagged words  : {len(tag_list):,}")
print(f"Unique tags   : {len(tag_freq)}")
print("\nPOS tag counts:")
print(
    tabulate(
        sorted(tag_freq.items()),
        headers=["POS", "Count"],
        tablefmt="github",
        intfmt=",",
    )
)

Sentences     : 43,196
Tagged words  : 564,517
Unique tags   : 15

POS tag counts:
| POS   |   Count |
|-------|---------|
| abb   |     360 |
| adj   |  16,430 |
| adv   |  10,711 |
| conj  |  17,808 |
| fw    |   3,228 |
| int   |     672 |
| n     | 122,892 |
| num   |   5,942 |
| part  | 135,267 |
| ppm   |  86,490 |
| pron  |  20,413 |
| punc  |  54,108 |
| sb    |     272 |
| tn    |   5,844 |
| v     |  84,080 |


## 6. Feature extraction for CRF

Word identity, length, prefix/suffix, digit/punctuation flags, and left/right context.

In [37]:
def extract_word_features(sent, i):
    w = sent[i][0]
    feats = {
        "bias": 1.0,
        "w": w,
        "w.len": len(w),
        "w.pref1": w[:1],
        "w.pref2": w[:2],
        "w.suf1": w[-1:],
        "w.suf2": w[-2:],
        "w.isdigit": w.isdigit(),
        "w.ispunct": all(not ch.isalnum() for ch in w),
    }

    if i > 0:
        prev = sent[i - 1][0]
        feats.update({
            "-1:w": prev,
            "-1:len": len(prev),
            "-1:pref1": prev[:1],
            "-1:suf1": prev[-1:],
        })
    else:
        feats["BOS"] = True

    if i < len(sent) - 1:
        nxt = sent[i + 1][0]
        feats.update({
            "+1:w": nxt,
            "+1:len": len(nxt),
            "+1:pref1": nxt[:1],
            "+1:suf1": nxt[-1:],
        })
    else:
        feats["EOS"] = True

    return feats


def sent_to_features(sent):
    return [extract_word_features(sent, i) for i in range(len(sent))]


def sent_to_labels(sent):
    return [tag for _, tag in sent]


def sent_to_words(sent):
    return [word for word, _ in sent]

In [38]:
print("Words :", sent_to_words(corpus[0]))
print("Labels:", sent_to_labels(corpus[0]))
print("\nFeatures (token 0):")
print(sent_to_features(corpus[0])[0])

Words : ['၁၉၆၂', 'ခုနှစ်', 'ခန့်မှန်း', 'သန်းခေါင်စာရင်း', 'အရ', 'လူဦးရေ', '၁၁၅၉၃၁', 'ယောက်', 'ရှိ', 'သည်', '။']
Labels: ['num', 'n', 'v', 'n', 'ppm', 'n', 'num', 'part', 'v', 'ppm', 'punc']

Features (token 0):
{'bias': 1.0, 'w': '၁၉၆၂', 'w.len': 4, 'w.pref1': '၁', 'w.pref2': '၁၉', 'w.suf1': '၂', 'w.suf2': '၆၂', 'w.isdigit': True, 'w.ispunct': False, 'BOS': True, '+1:w': 'ခုနှစ်', '+1:len': 6, '+1:pref1': 'ခ', '+1:suf1': '်'}


## 7. Train / test split (80% / 20%)

In [39]:
train_data, test_data = train_test_split(
    corpus, test_size=0.2, random_state=42
)

X_train = [sent_to_features(s) for s in train_data]
y_train = [sent_to_labels(s) for s in train_data]
X_test = [sent_to_features(s) for s in test_data]
y_test = [sent_to_labels(s) for s in test_data]

print(f"Train size: {len(train_data):,}")
print(f"Test size : {len(test_data):,}")

Train size: 34,556
Test size : 8,640


## 8. Train CRF model

In [40]:
pos_crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)

print("Training CRF POS tagger (please wait)...")
pos_crf.fit(X_train, y_train)
print("Training done.")
print("Label set:", sorted(pos_crf.classes_))

Training CRF POS tagger (please wait)...
Training done.
Label set: ['abb', 'adj', 'adv', 'conj', 'fw', 'int', 'n', 'num', 'part', 'ppm', 'pron', 'punc', 'sb', 'tn', 'v']


## 9. Testing

Run the trained model on the held-out test set.

In [41]:
print("Running prediction on test sentences...")
y_pred = pos_crf.predict(X_test)
print(f"Predicted tags for {len(y_pred):,} sentences.")

# Quick visual check
print("\n=== Sample test output ===")
sample_i = 0
words = sent_to_words(test_data[sample_i])
gold = y_test[sample_i]
pred = y_pred[sample_i]
print(
    tabulate(
        list(zip(words, gold, pred)),
        headers=["Word", "Gold", "Pred"],
        tablefmt="github",
    )
)

Running prediction on test sentences...
Predicted tags for 8,640 sentences.

=== Sample test output ===
| Word   | Gold   | Pred   |
|--------|--------|--------|
| အားနာ  | v      | v      |
| ပေမယ့်   | conj   | conj   |
| အဲဒီ     | adj    | pron   |
| ကိစ္စ    | n      | n      |
| ကို      | ppm    | ppm    |
| ကျွန်တော် | pron   | pron   |
| တာဝန်   | n      | n      |
| မ      | part   | part   |
| ယူ      | v      | v      |
| ပါ     | part   | part   |
| ဘူး     | part   | part   |
| ။      | punc   | punc   |


## 10. Evaluation

Accuracy, weighted Precision / Recall / F1, and per-tag classification report.

In [42]:
label_names = sorted(pos_crf.classes_)

acc = metrics.flat_accuracy_score(y_test, y_pred)
prec = metrics.flat_precision_score(
    y_test, y_pred, average="weighted", labels=label_names, zero_division=0
)
rec = metrics.flat_recall_score(
    y_test, y_pred, average="weighted", labels=label_names, zero_division=0
)
f1 = metrics.flat_f1_score(
    y_test, y_pred, average="weighted", labels=label_names, zero_division=0
)

print("=== Overall Evaluation ===")
print(
    tabulate(
        [
            ["Accuracy", f"{acc * 100:.2f}%"],
            ["Weighted Precision", f"{prec * 100:.2f}%"],
            ["Weighted Recall", f"{rec * 100:.2f}%"],
            ["Weighted F1-score", f"{f1 * 100:.2f}%"],
        ],
        headers=["Metric", "Score"],
        tablefmt="github",
    )
)

=== Overall Evaluation ===
| Metric             | Score   |
|--------------------|---------|
| Accuracy           | 96.02%  |
| Weighted Precision | 96.00%  |
| Weighted Recall    | 96.02%  |
| Weighted F1-score  | 96.00%  |


In [43]:
print("=== Per-tag Classification Report ===")
print(
    metrics.flat_classification_report(
        y_test,
        y_pred,
        labels=label_names,
        digits=4,
        zero_division=0,
    )
)

=== Per-tag Classification Report ===
              precision    recall  f1-score   support

         abb     0.9608    0.7101    0.8167        69
         adj     0.8486    0.8019    0.8246      3292
         adv     0.9128    0.8362    0.8728      2167
        conj     0.8941    0.9286    0.9110      3474
          fw     0.9752    0.9849    0.9800       598
         int     0.9470    0.9124    0.9294       137
           n     0.9579    0.9701    0.9640     24227
         num     0.9991    0.9957    0.9974      1174
        part     0.9652    0.9667    0.9659     26702
         ppm     0.9798    0.9834    0.9816     17029
        pron     0.9629    0.9610    0.9619      4021
        punc     0.9988    0.9994    0.9991     10782
          sb     1.0000    0.8103    0.8952        58
          tn     0.9793    0.9667    0.9729      1172
           v     0.9459    0.9384    0.9421     16611

    accuracy                         0.9602    111513
   macro avg     0.9552    0.9177    0.934

## 11. Demo: tag a custom Myanmar sentence

In [44]:
def predict_pos(words, model):
    dummy = [(w, "_") for w in words]
    return list(zip(words, model.predict_single(sent_to_features(dummy))))


demo_words = [
    "ကျွန်တော်",
    "သည်",
    "တက္ကသိုလ်",
    "သို့",
    "သွား",
    "နေ",
    "သည်",
    "။",
]

demo_tags = predict_pos(demo_words, pos_crf)
print(tabulate(demo_tags, headers=["Word", "Predicted POS"], tablefmt="github"))

| Word   | Predicted POS   |
|--------|-----------------|
| ကျွန်တော် | pron            |
| သည်     | ppm             |
| တက္ကသိုလ်  | n               |
| သို့      | ppm             |
| သွား    | v               |
| နေ     | part            |
| သည်     | ppm             |
| ။      | punc            |


## 12. Error analysis

Collect words where gold tag ≠ predicted tag.

In [45]:
mistakes = []
for sent, gold_tags, pred_tags in zip(test_data, y_test, y_pred):
    for word, g, p in zip(sent_to_words(sent), gold_tags, pred_tags):
        if g != p:
            mistakes.append([word, g, p])

print(f"Incorrect predictions: {len(mistakes):,}")
print("\nFirst 20 errors:")
print(
    tabulate(
        mistakes[:20],
        headers=["Word", "True Tag", "Predicted Tag"],
        tablefmt="github",
    )
)

Incorrect predictions: 4,436

First 20 errors:
| Word     | True Tag   | Predicted Tag   |
|----------|------------|-----------------|
| အဲဒီ       | adj        | pron            |
| ‌တော်တော်   | adv        | n               |
| အေး      | int        | part            |
| တုန်း      | part       | conj            |
| ကြား     | v          | n               |
| သွား      | v          | part            |
| နဲ့        | ppm        | conj            |
| နှစ်       | n          | tn              |
| ထောင်     | tn         | n               |
| ခမ်းခြောက် | v          | n               |
| သွား      | part       | v               |
| နဲ့        | part       | conj            |
| ချိတ်ဆက်    | n          | v               |
| တွင်       | part       | ppm             |
| အကြား    | ppm        | n               |
| နီ        | adj        | n               |
| လျှို့ဝှက်     | v          | adj             |
| ပိုင်း      | n          | part            |
| တန်းတူ     | v          | n           